In [2]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
import joblib

# Optional: XGBoost
from xgboost import XGBClassifier

OUT_MODEL_DIR = "../outputs/models/"
OUT_REPORT_DIR = "../outputs/reports/"
os.makedirs(OUT_MODEL_DIR, exist_ok=True)
os.makedirs(OUT_REPORT_DIR, exist_ok=True)

In [3]:
# --- cleaning functions (must match notebook 03 exactly) ---

NUMERIC_FEATURES = [
    "duration","src_bytes","dst_bytes","src_pkts","dst_pkts",
    "src_ip_bytes","dst_ip_bytes","missed_bytes","src_port","dst_port",
    "http_request_body_len","http_response_body_len"
]

CATEGORICAL_FEATURES = [
    "proto","service","conn_state","http_method","http_version",
    "http_status_code","ssl_version","ssl_cipher","weird_name"
]

BOOLEAN_FEATURES = ["dns_AA","dns_RD","dns_RA","ssl_resumed","ssl_established"]

DASH_AS_NONE_COLS = CATEGORICAL_FEATURES + [
    "dns_query", "http_uri", "http_user_agent",
    "http_orig_mime_types", "http_resp_mime_types",
    "weird_addl", "weird_notice",
    "ssl_subject", "ssl_issuer"
]

def normalize_dash(df_):
    df_ = df_.copy()
    cols = [c for c in DASH_AS_NONE_COLS if c in df_.columns]
    for c in cols:
        df_[c] = df_[c].astype(str).replace("-", "NONE")
    return df_

def cast_boolean_like_df(df_, bool_cols):
    df_ = df_.copy()
    for col in bool_cols:
        s = df_[col]
        if pd.api.types.is_numeric_dtype(s):
            df_[col] = s.fillna(0).astype(int)
            continue
        s = s.astype(str).replace("-", "NONE").str.lower()
        true_set = {"t", "true", "1", "yes"}
        df_[col] = s.apply(lambda v: 1 if v in true_set else 0).astype(int)
    return df_


In [4]:
import numpy as np
import json

X_train = np.load("../outputs/processed/X_train.npy")
y_train = np.load("../outputs/processed/y_train.npy")

X_val = np.load("../outputs/processed/X_val.npy")
y_val = np.load("../outputs/processed/y_val.npy")

X_test = np.load("../outputs/processed/X_test.npy")
y_test = np.load("../outputs/processed/y_test.npy")

with open("../outputs/processed/feature_order.json", "r", encoding="utf-8") as f:
    FEATURE_NAMES = json.load(f)

print("Loaded processed arrays:")
print("Train:", X_train.shape, y_train.shape)
print("Val:",   X_val.shape,   y_val.shape)
print("Test:",  X_test.shape,  y_test.shape)
print("n_features:", len(FEATURE_NAMES))


Loaded processed arrays:
Train: (2345597, 104) (2345597,)
Val: (502628, 104) (502628,)
Test: (502628, 104) (502628,)
n_features: 104


In [5]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

RANDOM_STATE = 42
SUBSET_N = 400_000  # 200k-600k đều hợp lý

sss = StratifiedShuffleSplit(n_splits=1, train_size=SUBSET_N, random_state=RANDOM_STATE)
idx_subset, _ = next(sss.split(np.zeros(len(y_train)), y_train))

X_train_sub = np.asarray(X_train[idx_subset], dtype=np.float32)  # chỉ load subset vào RAM
y_train_sub = np.asarray(y_train[idx_subset], dtype=np.int64)

print("Subset:", X_train_sub.shape, y_train_sub.shape)
print("Label % attack:", y_train_sub.mean())


Subset: (400000, 104) (400000,)
Label % attack: 0.9643675


In [6]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train_sub, y_train_sub)

joblib.dump(xgb_model, f"{OUT_MODEL_DIR}/xgboost_baseline.pkl")
print("Saved XGBoost baseline model.")


Saved XGBoost baseline model.


In [7]:
logreg_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=1,
    random_state=42
)
logreg_model.fit(X_train_sub, y_train_sub)

joblib.dump(logreg_model, f"{OUT_MODEL_DIR}/logreg_baseline.pkl")
print("Saved Logistic Regression baseline model.")


Saved Logistic Regression baseline model.


In [8]:
def evaluate_model(model, X, y, name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_prob),
        "pr_auc": average_precision_score(y, y_prob),
    }

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    metrics["false_negative_rate"] = fn / (fn + tp)

    print(f"\n{name} evaluation:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    return metrics


In [9]:
metrics_xgb = evaluate_model(xgb_model, X_test, y_test, "XGBoost")
metrics_logreg = evaluate_model(logreg_model, X_test, y_test, "Logistic Regression")



XGBoost evaluation:
accuracy: 0.9991
precision: 0.9994
recall: 0.9996
f1: 0.9995
roc_auc: 0.9999
pr_auc: 1.0000
false_negative_rate: 0.0004

Logistic Regression evaluation:
accuracy: 0.9342
precision: 0.9965
recall: 0.9350
f1: 0.9648
roc_auc: 0.9721
pr_auc: 0.9987
false_negative_rate: 0.0650


In [10]:
all_metrics = {
    "xgboost": metrics_xgb,
    "logistic_regression": metrics_logreg
}

with open(f"{OUT_REPORT_DIR}/baseline_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

print("Saved baseline metrics to outputs/reports/")


Saved baseline metrics to outputs/reports/
